# Create edgotype profiles

In this step, we make one edgotype profile per gene of interactors by alleles. Later, we will explore metrics that summarize overall perturbation for each allele (quantitative score like imaging / abundance)

In [45]:
import polars as pl 

In [46]:
inputs_dir = "../1_inputs/sqY2H"
outputs_dir = "../3_outputs/sqY2H"

In [47]:
scores = pl.read_csv(f"{outputs_dir}/4_normalized_consensus_scores.csv")

# keep only scores where there is at least one matched reference-variant set
orfs_with_alleles = scores.filter(pl.col("db_mut_id") != 0).select("db_orf_id").unique().to_series().to_list()
orfs = scores.filter(pl.col("db_mut_id") == 0).select("db_orf_id").unique().to_series().to_list()
complete_orf_pairs = [i for i in orfs if i in orfs_with_alleles]
scores = scores.filter(pl.col("db_orf_id").is_in(complete_orf_pairs))

genes = scores.select("db_orf_id").to_series().unique().to_list()

In [48]:
# Create profiles for each profiled gene

profiles = {}
shape_summary = []
for gene in genes:
    gene_scores = scores.filter(pl.col("db_orf_id") == gene)
    profile = gene_scores.pivot(
                values="normalized_score",
                index="ad_orf_id",
                on="db_mut_id"
            )
    profiles[gene] = profile

    shape_summary.append({
        "gene": gene,
        "n_interactors": profile.height,
        "n_alleles": profile.width
    })

    profile.write_csv(f"{outputs_dir}/edgotype_profiles/orf_id_{gene}.csv")

shape_summary = pl.DataFrame(shape_summary)

In [52]:
## Add a perturbation category for each edge to use for the N2H validation

# Extract WT scores
wt_scores = scores.filter(pl.col("db_mut_id") == 0).select(["db_orf_id", "ad_orf_id", "normalized_score"])

# merge wt scores back on the main df
scores_with_wt = scores.join(wt_scores, on=["db_orf_id", "ad_orf_id"], how="left", suffix="_wt")

# compute allele/wt ratio
scores_with_wt = scores_with_wt.with_columns(
    (pl.col("normalized_score") / pl.col("normalized_score_wt")).alias("ratio")
)

# Determine if the allele is perturbed
threshold = 0.5
scores_with_wt = scores_with_wt.with_columns(
    ((pl.col("db_mut_id") != 0) & (pl.col("ratio") < threshold)).alias("perturbed")
)
# Final dataframe with the "perturbed" column
scores_with_perturbed = scores_with_wt.select("*")

cols_to_keep = ['ad_orf_id',
 'db_orf_id',
 'db_mut_id',
 'normalized_score',
 'ad_symbol',
 'symbol',
 'clinvar_clnsig_clean',
 'normalized_score_wt',
 'ratio',
 'perturbed']

scores_with_perturbed = scores_with_perturbed.select(cols_to_keep)

scores_with_perturbed.write_csv(f"{outputs_dir}/7_edge_perturbation_categorical.csv")

## Examine profile characteristics

In [50]:
# Number of profiled alleles with more than 5 interactors
hq_orfs = shape_summary.filter(pl.col("n_interactors") >= 5)
num_orfs = hq_orfs.shape[0]
print(num_orfs)

print(hq_orfs.select(pl.col("n_alleles")).sum() - num_orfs)

26
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 295       │
└───────────┘


In [51]:
# Total number of profiled alleles
prof_genes = shape_summary.shape[0]
print(prof_genes)

print(shape_summary.select(pl.col("n_alleles")).sum() - prof_genes)

84
shape: (1, 1)
┌───────────┐
│ n_alleles │
│ ---       │
│ i64       │
╞═══════════╡
│ 800       │
└───────────┘
